# PVDAQ System 50 — Data-Usability Pipeline for RUL Modelling

**Goal.** Look at *every* year of System 50 data, decide **which days (and therefore which years) are good enough to train a Remaining-Useful-Life (RUL) model**, and save the good part in a tidy format that the next notebooks can load directly.

**Why this notebook exists.** Our earlier tests showed that the raw data cannot be used as-is: some timestamps are recorded twice with different values, the irradiance sensor (`poa_irradiance`) is unreliable, and 2023 — especially **February 2023 (the Colorado snowstorm)** — is damaged. So instead of guessing, this pipeline measures the quality of **every single day** and then summarises it per year.

### How the pipeline follows the ML-pipeline rules
This notebook covers the **first three steps only**:

| Step | What happens here | Notebook part |
|---|---|---|
| 1. Data inspection | list every file, profile every day | Part 1 |
| 2. Exploratory analysis | scorecards, calendar, snow detection, irradiance validity, degradation signal | Part 2 |
| 3. Define the prediction problem | draft contract for you to approve | Part 3 |
| **GATE A** | **you approve the contract** (a switch in the notebook) | end of Part 3 |
| 4–5. Cleaning + data engineering | write the RUL-ready tables | Part 4 — **locked until Gate A** |

No model is trained anywhere in this notebook.

### What you get at the end
All files go to the `ml_pipeline/` folder next to the notebook (`/kaggle/working/ml_pipeline/` on Kaggle):

| File | Written | Contents |
|---|---|---|
| `data/daily_quality.parquet` | always | one row per calendar day: every quality measure + the reason a day was kept or dropped |
| `data/year_scorecard.csv` | always | one row per year: how many usable days, verdict `USE / USE_WITH_CAUTION / DO_NOT_USE` |
| `data_profile.json`, `figures/*.png`, `PIPELINE.md` | always | evidence for the Gate-A review |
| `data/rul_ready_daily.parquet` | **after Gate A** | usable days only, one row per day, clean features for RUL work |
| `data/minute_level/...` | after Gate A, optional | minute-level cleaned data for chosen years |
| `data/manifest.json` | after Gate A | settings, counts, library versions — for reproducibility |

**Running time.** Profiling every day of ~31 years means reading ~11,000 small files from S3. Expect tens of minutes with internet on. Results are cached per year, so if Kaggle stops the session you can re-run and it continues where it stopped. For a quick trial, set `DAY_STRIDE = 7` (every 7th day).

## Part 0 — Setup and settings

**What this cell does.** Loads the libraries and defines every setting in one place: which system, where to save files, and the thresholds that decide whether a day is "usable". Nothing is read from the internet yet.

**Why it matters.** All judgement calls live here, so you can see and change them. The values below are sensible starting points, not truths — the figures later show how much each one matters.

In [ ]:
import json, hashlib, sys, subprocess, warnings, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import s3fs

try:
    import pvlib
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pvlib"], check=True)
    import pvlib

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# ---------------- what to process ----------------
SYSTEM_ID = 50
YEARS = None            # None = every year found on S3, or e.g. [2018, 2019, 2020]
DAY_STRIDE = 1          # 1 = every day. 7 = every 7th day (quick scouting run)
WORKERS = 16            # parallel S3 downloads
FORCE_REPROFILE = False # True = ignore cached per-year results

# ---------------- site and clock ----------------
LAT, LON, ALT = 39.7407, -105.1686, 1795   # NREL South Table Mountain, Golden CO
TILT, AZIMUTH = 40.0, 180.0                # replace with mount-table values if you have them
SUN_ELEV_DAY = 10.0                        # sun higher than this (degrees) = "daylight"

# ---------------- which signal is "power" ----------------
POWER_CHANNEL = "dc_power"
TEMP_COEFF = -0.004                        # typical silicon: power falls 0.4 % per degree C above 25 C

# ---------------- day-level usability rules ----------------
MIN_DAYLIGHT_COV = 0.90      # power must be logged for at least 90 % of daylight minutes
MAX_DAYLIGHT_GAP_MIN = 60    # no daylight hole longer than 60 minutes
MAX_CONFLICT_SHARE = 0.05    # at most 5 % of daylight minutes touched by conflicting duplicates
MIN_CLEAR_MINUTES = 30       # need at least 30 clear-sky minutes to compute a performance index

# ---------------- snow / outage detection ----------------
SNOW_ENERGY_RATIO = 0.30     # day produced < 30 % of what a normal day should
SNOW_AMBIENT_C = 2.0         # and the daytime air was colder than this
SNOW_MIN_RUN_DAYS = 2        # for at least this many days in a row
SNOW_PAD_DAYS = 1            # also drop this many days after (melting)

# ---------------- known bad periods (human knowledge beats statistics) ----------------
EXCLUDE_PERIODS = [
    ("2023-02-01", "2023-02-28", "Colorado snowstorm - damaged data (confirmed by project owner)"),
]
EXCLUDE_YEARS = []           # e.g. [2023] to drop a whole year

# ---------------- irradiance sensor validity (day level) ----------------
POA_MIN_DAY_COV = 0.80       # logged for >= 80 % of daylight minutes
POA_DAY_MEDIAN_RANGE = (20, 1400)   # W/m2, daytime median must be physical
POA_NIGHT_MEDIAN_RANGE = (-20, 30)  # W/m2, night-time reading must be about zero
POA_MIN_CORR = 0.70          # must follow power during the day

# ---------------- year verdict ----------------
USE_MIN_DAYS, USE_MIN_MONTHS = 150, 9
CAUTION_MIN_DAYS = 60

# ---------------- RUL contract draft (only used in Part 3) ----------------
EOL_FRACTION = 0.80          # "end of life" = health falls to 80 % of its starting level

# ---------------- output folders ----------------
ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT_DIR = ROOT / "ml_pipeline"
FIG_DIR = OUT_DIR / "figures"
DATA_DIR = OUT_DIR / "data"
CACHE_DIR = DATA_DIR / "cache"
for d in (FIG_DIR, DATA_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

GATE_A_APPROVED = False      # <-- set to True ONLY after you approve the contract in Part 3

BUCKET = "oedi-data-lake"
BASE = f"{BUCKET}/pvdaq/parquet"
KEYS = ["measured_on", "metric_id"]
fs = s3fs.S3FileSystem(anon=True)
print("Output folder:", OUT_DIR.resolve())

# Part 1 — Data inspection (step 1)

Nothing is changed in this part. We only look and measure.

## 1.1 Sensor name lookup

**What this cell does.** Downloads the small `metrics` table and builds two dictionaries: `NAME` (metric number → sensor name, e.g. `751 → dc_power`) and `ID` (the reverse).

**Why it matters.** The raw files only contain numbers like `751`. We need the names to know what each number means. The cell stops with an error if the power channel we chose in the settings does not exist, so we fail early instead of late.

In [ ]:
metrics = pd.read_parquet(f"{BASE}/metrics", filesystem=fs)
metrics_50 = metrics[metrics["system_id"].eq(SYSTEM_ID)].sort_values("metric_id")
assert metrics_50["metric_id"].is_unique, "metric_id is not unique - inspect the metadata first"

NAME = metrics_50.set_index("metric_id")["sensor_name"].to_dict()
ID = {v: k for k, v in NAME.items()}
assert POWER_CHANNEL in ID, f"{POWER_CHANNEL} not found for system {SYSTEM_ID}"
print(len(NAME), "sensors:", list(NAME.values()))

## 1.2 Inventory of every file on S3

**What this cell does.** Asks S3 for the list of all data files of System 50 (without downloading them), reads the year / month / day out of each path, and groups the files by day.

**Why it matters.** This tells us *which days exist at all*. A day with no file is a hole in the history, and a day with two files is a hint of duplicate ingestion. Files whose names start with `_` or `.` (bookkeeping files) are ignored.

In [ ]:
def build_inventory():
    root = f"{BASE}/pvdata/system_id={SYSTEM_ID}"
    info = fs.find(root, detail=True)
    rows = [(p, v.get("size", 0)) for p, v in info.items()
            if v.get("type") != "directory" and v.get("size", 0) > 0]
    inv = pd.DataFrame(rows, columns=["path", "size"])
    inv = inv[~inv["path"].str.rsplit("/", n=1).str[-1].str.startswith(("_", "."))]
    ex = inv["path"].str.extract(r"year=(\d+)/month=(\d+)/day=(\d+)")
    ok = ex.notna().all(axis=1)
    print("Files ignored (no year/month/day in path):", int((~ok).sum()))
    inv = inv[ok].copy()
    inv[["year", "month", "day"]] = ex[ok].astype(int).to_numpy()
    inv["date"] = pd.to_datetime(inv[["year", "month", "day"]])
    inv["day_dir"] = inv["path"].str.rsplit("/", n=1).str[0]
    return inv.sort_values("path", ignore_index=True)

inventory = build_inventory()
days = (inventory.groupby("day_dir")
        .agg(date=("date", "first"), n_files=("path", "size"),
             mb=("size", lambda s: s.sum() / 1e6), paths=("path", list))
        .reset_index(drop=True).sort_values("date", ignore_index=True))

if YEARS is not None:
    days = days[days["date"].dt.year.isin(YEARS)]
if DAY_STRIDE > 1:
    days = days.iloc[::DAY_STRIDE]
days = days.reset_index(drop=True)

print(f"Days with files: {len(days):,}   first: {days['date'].min().date()}   last: {days['date'].max().date()}")
print("Total size (MB):", round(days["mb"].sum(), 1))
print("Days with more than one file:", int((days["n_files"] > 1).sum()))
display(days.groupby(days["date"].dt.year).agg(days=("date", "size"), files=("n_files", "sum"),
                                              mb=("mb", "sum")).round(1).T)

## 1.3 The per-day profiler

**What this cell does.** Defines the function that looks at **one day** of data and returns one row of quality measurements. It is the heart of the pipeline. For one day it:

1. Reads the day's file(s) and counts rows.
2. Finds **repeated readings**: the same sensor at the same minute more than once. If the copies are identical they are harmless; if they differ they are a **conflict**.
3. Builds a clean one-minute table. Identical copies are collapsed to one. **Conflicting readings are set to "missing" — never averaged** — because an average would be a number the sensor never reported.
4. Works out where the sun is, minute by minute, so we know which minutes are daylight.
5. Measures how complete the power signal is in daylight, and the longest hole.
6. Checks whether the irradiance sensor (POA) is believable.
7. Computes a **performance index**: power divided by the power the sun *should* give on a clear day (from a clear-sky model), corrected for module temperature. This is our candidate "health" signal, and it does **not** need the unreliable irradiance sensor.

**Why it matters.** Every later decision (which days, which years) is based on these numbers, so they are computed the same way for every day.

In [ ]:
CH_KEY = ["poa_irradiance", POWER_CHANNEL, "ac_power", "module_temp_1", "ambient_temp"]
MINUTES = 1440

def longest_run(a):
    # length of the longest streak of True values
    a = np.asarray(a, dtype=np.int8)
    if not a.any():
        return 0
    d = np.diff(np.concatenate(([0], a, [0])))
    return int((np.where(d == -1)[0] - np.where(d == 1)[0]).max())

def read_day(paths):
    frames = []
    for k, p in enumerate(paths):
        d = pd.read_parquet(p, filesystem=fs, columns=["measured_on", "utc_measured_on", "metric_id", "value"])
        d["file_no"] = k
        frames.append(d)
    return pd.concat(frames, ignore_index=True)

def clean_day(raw, date):
    # Returns: wide 1-minute table (conflicts -> NaN), per-minute info, and counts.
    g = raw.groupby(KEYS)["value"].agg(n="size", nu="nunique")
    conflict_keys = g[g["nu"] > 1]
    conf_per_ts = pd.Series(1, index=conflict_keys.index.get_level_values("measured_on")).groupby(level=0).size()

    first = raw.drop_duplicates(KEYS, keep="first").merge(g["nu"].reset_index(), on=KEYS, how="left")
    first.loc[first["nu"] > 1, "value"] = np.nan          # conflicts become missing, never averaged

    wide = first.pivot(index="measured_on", columns="metric_id", values="value")
    wide.columns = [NAME.get(c, f"metric_{c}") for c in wide.columns]

    idx = pd.date_range(date, periods=MINUTES, freq="1min")
    n_outside = int((~wide.index.isin(idx)).sum())         # rows stamped outside this calendar day
    wide = wide.reindex(idx)

    off = (raw["utc_measured_on"] - raw["measured_on"]).mode().iloc[0]
    idx_utc = pd.DatetimeIndex(idx + off).tz_localize("UTC")
    sp = pvlib.solarposition.get_solarposition(idx_utc, LAT, LON, altitude=ALT)

    info = pd.DataFrame({"elev": sp["apparent_elevation"].to_numpy(),
                         "zen": sp["apparent_zenith"].to_numpy(),
                         "az": sp["azimuth"].to_numpy(),
                         "conflict_minute": idx.isin(conf_per_ts.index)}, index=idx)
    info["n_conflict_metrics"] = conf_per_ts.reindex(idx).fillna(0).astype(int).to_numpy()

    counts = {"n_rows": len(raw),
              "n_timestamps": int(raw["measured_on"].nunique()),
              "n_files": int(raw["file_no"].nunique()),
              "n_ts_outside_day": n_outside,
              "utc_offset_h": off.total_seconds() / 3600,
              "exact_dup_groups": int(((g["n"] > 1) & (g["nu"] == 1)).sum()),
              "conflict_groups": int((g["nu"] > 1).sum()),
              "conflict_minutes_any": int(len(conf_per_ts)),
              "conflict_minutes_broad": int((conf_per_ts >= 7).sum())}
    return wide, info, idx_utc, counts

def clearsky_poa(info, idx_utc):
    loc = pvlib.location.Location(LAT, LON, tz="UTC", altitude=ALT)
    cs = loc.get_clearsky(idx_utc, model="ineichen")
    poa = pvlib.irradiance.get_total_irradiance(TILT, AZIMUTH, info["zen"].to_numpy(), info["az"].to_numpy(),
                                                cs["dni"].to_numpy(), cs["ghi"].to_numpy(), cs["dhi"].to_numpy())
    return pd.Series(np.asarray(poa["poa_global"]), index=info.index)

def profile_day(date, paths):
    row = {"date": date, "year": date.year, "month": date.month, "doy": date.dayofyear}
    try:
        raw = read_day(paths)
        wide, info, idx_utc, counts = clean_day(raw, date)
        row.update(counts)

        def col(n):
            return wide[n] if n in wide else pd.Series(np.nan, index=wide.index)

        P, Q = col(POWER_CHANNEL), col("poa_irradiance")
        Tm, Ta = col("module_temp_1"), col("ambient_temp")
        Tm = Tm.where(Tm.between(-50, 100)); Ta = Ta.where(Ta.between(-50, 60))
        day = (info["elev"] > SUN_ELEV_DAY).to_numpy()
        night = (info["elev"] < -6).to_numpy()
        D = int(day.sum())
        row["daylight_minutes"] = D

        # --- completeness of power in daylight
        row["cov_day_power"] = float(P[day].notna().mean()) if D else np.nan
        row["max_daylight_gap_min"] = longest_run(P.isna().to_numpy() & day)
        row["conflict_share_day"] = float(info["conflict_minute"].to_numpy()[day].mean()) if D else np.nan
        row["power_flatline"] = bool(D and P[day].nunique() <= 3)
        row["dc_peak_w"] = float(P.quantile(0.999)) if P.notna().any() else np.nan
        row["dc_energy_kwh_obs"] = float(P.clip(lower=0).sum() / 60 / 1000)
        row["power_neg_day_frac"] = float((P[day] < -1).sum() / max(1, P[day].notna().sum())) if D else np.nan

        # --- temperatures
        row["module_temp_day_mean"] = float(Tm[day].mean()) if D else np.nan
        row["ambient_day_mean"] = float(Ta[day].mean()) if D else np.nan
        row["ambient_min"] = float(Ta.min()) if Ta.notna().any() else np.nan

        # --- irradiance sensor validity
        row["poa_cov_day"] = float(Q[day].notna().mean()) if D else np.nan
        row["poa_cov_night"] = float(Q[night].notna().mean()) if night.any() else np.nan
        row["poa_day_median"] = float(Q[day].median()) if D else np.nan
        row["poa_night_median"] = float(Q[night].median()) if night.any() else np.nan
        both = Q[day].notna() & P[day].notna()
        row["poa_corr_power_day"] = float(Q[day][both].corr(P[day][both])) if both.sum() >= 30 else np.nan
        row["poa_valid_day"] = bool(
            row["poa_cov_day"] >= POA_MIN_DAY_COV
            and POA_DAY_MEDIAN_RANGE[0] <= row["poa_day_median"] <= POA_DAY_MEDIAN_RANGE[1]
            and POA_NIGHT_MEDIAN_RANGE[0] <= (row["poa_night_median"] if pd.notna(row["poa_night_median"]) else 0) <= POA_NIGHT_MEDIAN_RANGE[1]
            and (row["poa_corr_power_day"] if pd.notna(row["poa_corr_power_day"]) else 0) >= POA_MIN_CORR
        ) if D and pd.notna(row["poa_day_median"]) else False

        # --- clear-sky performance index (does not use the POA sensor)
        poa_cs = clearsky_poa(info, idx_utc)
        row["cs_energy_kwh_m2"] = float(poa_cs.sum() / 60 / 1000)
        ok = (info["elev"] > 15) & (poa_cs > 300) & (P > 0)
        r = (P / poa_cs)[ok]
        row["clear_minutes"] = 0
        row["perf_index_raw"] = np.nan
        row["perf_index_tc"] = np.nan
        if len(r) >= 20:
            cv = r.rolling("15min").std() / r.rolling("15min").mean()
            clear = (cv < 0.05) & (r > 0.8 * r.quantile(0.9))
            row["clear_minutes"] = int(clear.sum())
            if clear.sum() >= 5:
                tc = 1 + TEMP_COEFF * (Tm[ok] - 25.0)
                row["perf_index_raw"] = float(r[clear].median())
                row["perf_index_tc"] = float((r / tc)[clear].median()) if (tc[clear].notna().any()) else np.nan
    except Exception as e:
        row["read_error"] = f"{type(e).__name__}: {e}"[:200]
    return row

print("profile_day ready")

## 1.4 Run the profiler over every year

**What this cell does.** Runs `profile_day` on every day, many days at a time in parallel, one year after another. After each year it saves a small cache file. If the session stops, re-running continues from the last finished year.

**Why it matters.** This is the slow cell. The output, `daily`, is a compact table with **one row per day** (a few thousand rows) instead of billions of sensor readings, which makes everything after this fast.

In [ ]:
CFG_KEYS = dict(LAT=LAT, LON=LON, ALT=ALT, TILT=TILT, AZIMUTH=AZIMUTH, SUN_ELEV_DAY=SUN_ELEV_DAY,
                POWER_CHANNEL=POWER_CHANNEL, TEMP_COEFF=TEMP_COEFF, DAY_STRIDE=DAY_STRIDE,
                POA=[POA_MIN_DAY_COV, POA_DAY_MEDIAN_RANGE, POA_NIGHT_MEDIAN_RANGE, POA_MIN_CORR])
CFG_HASH = hashlib.md5(json.dumps(CFG_KEYS, sort_keys=True, default=str).encode()).hexdigest()[:8]

def profile_year(year, days_y):
    cache = CACHE_DIR / f"daily_profile_{year}_{CFG_HASH}.parquet"
    if cache.exists() and not FORCE_REPROFILE:
        return pd.read_parquet(cache), True
    with ThreadPoolExecutor(WORKERS) as ex:
        rows = list(ex.map(lambda t: profile_day(t[0], t[1]), zip(days_y["date"], days_y["paths"])))
    out = pd.DataFrame(rows)
    out.to_parquet(cache, index=False)
    return out, False

t0 = time.time()
parts = []
for year, days_y in days.groupby(days["date"].dt.year):
    df_y, cached = profile_year(year, days_y)
    parts.append(df_y)
    errs = int(df_y["read_error"].notna().sum()) if "read_error" in df_y else 0
    print(f"{year}: {len(df_y):4d} days  errors={errs:3d}  {'(cached)' if cached else ''}  elapsed {time.time() - t0:6.0f}s")

profiled = pd.concat(parts, ignore_index=True).sort_values("date", ignore_index=True)
if "read_error" not in profiled:
    profiled["read_error"] = None
print("\nProfiled days:", len(profiled), "| days with read errors:", int(profiled["read_error"].notna().sum()))
display(profiled.head(3).T)

## 1.5 Fill in the days that have no file

**What this cell does.** Builds a full calendar from the first to the last day and joins our profile onto it. Days with no file appear as empty rows labelled `has_file = False`.

**Why it matters.** A missing day is not "zero". By listing it explicitly, we can count how much of each year is really covered, and draw gaps in the calendar picture later.

In [ ]:
cal = pd.DataFrame({"date": pd.date_range(profiled["date"].min(), profiled["date"].max(), freq="D")})
daily = cal.merge(profiled, on="date", how="left")
daily["has_file"] = daily["n_rows"].notna() & daily["read_error"].isna()
daily["year"] = daily["date"].dt.year
daily["month"] = daily["date"].dt.month
daily["doy"] = daily["date"].dt.dayofyear
print("Calendar days:", len(daily), "| days with usable files:", int(daily["has_file"].sum()),
      "| days without:", int((~daily["has_file"]).sum()))

## 1.6 Figure — how much history exists per year

**What this cell does.** Draws one bar per year showing how many days have a data file, with a line marking the days in the calendar year. Also defines a helper that saves every figure to disk together with a short plain-English explanation, so the evidence survives even if the notebook is closed.

**How to read it.** A bar that reaches the line means the year is fully covered. Short bars mean holes — the logger was off, or the files are missing.

In [ ]:
FIG_LOG = []

def save_fig(step, name, fig, explanation):
    fname = f"{step}_{name}.png"
    fig.savefig(FIG_DIR / fname, dpi=130, bbox_inches="tight")
    FIG_LOG.append({"file": fname, "explanation": explanation})
    plt.show()
    print("EXPLANATION:", explanation)

yr = daily.groupby("year").agg(calendar_days=("date", "size"), days_with_files=("has_file", "sum"))
fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(yr.index, yr["days_with_files"], color="#3b7ddd", label="days with a data file")
ax.plot(yr.index, yr["calendar_days"], color="black", lw=1.2, marker="_", ms=14, label="days in calendar year")
ax.set_xlabel("year"); ax.set_ylabel("days"); ax.set_title("1. Coverage: days with a data file per year")
ax.legend(); ax.grid(alpha=.3, axis="y"); plt.tight_layout()

full = int((yr["days_with_files"] >= 0.95 * yr["calendar_days"]).sum())
save_fig("01", "inventory_days_per_year", fig,
         f"{full} of {len(yr)} years have a data file for at least 95% of their days; the shortest year has only "
         f"{int(yr['days_with_files'].min())} days. A year with many missing days cannot give a continuous health history.")

# Part 2 — Exploratory analysis (step 2)

Now we use the day-level numbers to answer: **which days can we trust, and what is wrong with the rest?**

## 2.1 Snow / outage detection

**What this cell does.** For each day it computes a "normal output" benchmark: the 90th-percentile yield (energy produced per unit of clear-sky sunlight) over the surrounding year. A day's `energy_vs_expected` is its yield divided by that benchmark. A day is a **snow suspect** if it produced under 30 % of normal **and** the daytime air was colder than 2 °C. Suspect days that last **2 or more days in a row** are marked as a **snow episode**, plus one extra day for melting.

**Why it matters.** Snow on the panels (or an outage) makes the power collapse even though nothing is wrong with the panels. If we left those days in, a model would mistake snow for ageing. Single cloudy days are *not* flagged, because clouds do not usually last in cold weather for several days with near-zero output. The February 2023 storm is also excluded by hand in the settings, so we are covered either way.

In [ ]:
d = daily.copy()
good_cov = d["cov_day_power"] >= MIN_DAYLIGHT_COV
d["yield_idx"] = (d["dc_energy_kwh_obs"] / d["cs_energy_kwh_m2"]).where(good_cov & (d["cs_energy_kwh_m2"] > 0))

ref = (d.set_index("date")["yield_idx"]
         .rolling("365D", center=True, min_periods=30)
         .quantile(0.90))
d["yield_ref"] = ref.to_numpy()
d["energy_vs_expected"] = d["yield_idx"] / d["yield_ref"]

d["snow_suspect"] = ((d["energy_vs_expected"] < SNOW_ENERGY_RATIO) & (d["ambient_day_mean"] < SNOW_AMBIENT_C)).fillna(False)
grp = (d["snow_suspect"] != d["snow_suspect"].shift()).cumsum()
runlen = d["snow_suspect"].groupby(grp).transform("size")
episode = d["snow_suspect"] & (runlen >= SNOW_MIN_RUN_DAYS)
d["snow_episode"] = episode.astype(int).rolling(SNOW_PAD_DAYS + 1, min_periods=1).max().astype(bool)

print("Snow-suspect days:", int(d["snow_suspect"].sum()), "| days inside snow episodes (with padding):", int(d["snow_episode"].sum()))
ep = d[d["snow_episode"]].copy()
if len(ep):
    ep["ep_id"] = ((ep["date"].diff() != pd.Timedelta("1D"))).cumsum()
    eps = ep.groupby("ep_id").agg(start=("date", "min"), end=("date", "max"), days=("date", "size"),
                                  min_energy_ratio=("energy_vs_expected", "min"))
    display(eps.sort_values("days", ascending=False).head(15))
else:
    print("No multi-day snow episodes found.")

## 2.2 Apply the day-level usability rules

**What this cell does.** Gives every calendar day exactly **one verdict**. The rules are checked in the order below and the *first* failing rule becomes the reason:

1. `no_data` — no readable file
2. `manual_exclusion` — in a period or year we excluded by hand (Feb 2023)
3. `conflicts` — more than 5 % of daylight minutes touched by conflicting duplicates (checked before coverage, because conflicting readings are turned into gaps and would otherwise be blamed on "low coverage")
4. `low_daylight_coverage` — power logged for less than 90 % of daylight minutes
5. `long_gap` — a daylight hole of more than 60 minutes
6. `flatline_or_bad_power` — power is frozen or never positive
7. `snow_episode` — flagged by the snow detector
8. `no_clear_minutes` — too few clear-sky minutes to compute the performance index

A day that passes all eight is `usable`. The cell also prints a **before/after table**: how many days are left after each rule.

**Why it matters.** We keep the data we can trust and we always know *why* a day was dropped. We do not delete rows — we label them — so every decision can be reviewed or changed later.

In [ ]:
REASONS = ["usable", "no_data", "manual_exclusion", "conflicts", "low_daylight_coverage", "long_gap",
           "flatline_or_bad_power", "snow_episode", "no_clear_minutes"]

manual = pd.Series(False, index=d.index)
manual_why = pd.Series("", index=d.index)
for start, end, why in EXCLUDE_PERIODS:
    m = d["date"].between(pd.Timestamp(start), pd.Timestamp(end))
    manual |= m
    manual_why = manual_why.mask(m, why)
if EXCLUDE_YEARS:
    m = d["year"].isin(EXCLUDE_YEARS)
    manual |= m
    manual_why = manual_why.mask(m, "whole year excluded in settings")
d["manual_exclusion"] = manual
d["manual_exclusion_why"] = manual_why

conds = [
    ~d["has_file"],
    d["manual_exclusion"],
    ~(d["conflict_share_day"] <= MAX_CONFLICT_SHARE),
    ~(d["cov_day_power"] >= MIN_DAYLIGHT_COV),
    ~(d["max_daylight_gap_min"] <= MAX_DAYLIGHT_GAP_MIN),
    d["power_flatline"].fillna(True).astype(bool) | ~(d["dc_peak_w"] > 0),
    d["snow_episode"],
    ~(d["clear_minutes"] >= MIN_CLEAR_MINUTES),
]
d["reason"] = np.select(conds, REASONS[1:], default="usable")
d["usable"] = d["reason"].eq("usable")
d["reason_code"] = d["reason"].map({r: i for i, r in enumerate(REASONS)})

remaining = len(d)
wf = [("all calendar days", remaining)]
for r in REASONS[1:]:
    remaining -= int((d["reason"] == r).sum())
    wf.append((f"after rule: {r}", remaining))
waterfall = pd.DataFrame(wf, columns=["step", "days_left"])
waterfall["dropped_by_rule"] = -waterfall["days_left"].diff().fillna(0).astype(int)
display(waterfall)
print(f"Usable days: {int(d['usable'].sum()):,} of {len(d):,} ({d['usable'].mean() * 100:.1f}%)")

## 2.3 Figure — the usability calendar

**What this cell does.** Draws one row per year and one column per day of the year, coloured by the verdict from the previous cell. Green = usable; each other colour is a different reason.

**How to read it.** Big green areas are the good history. Blocks of one colour show a systematic problem (e.g. a stretch of `conflicts` in spring 2020, or the grey block for February 2023).

In [ ]:
years = sorted(d["year"].unique())
M = np.full((len(years), 366), np.nan)
for i, y in enumerate(years):
    s = d[d["year"] == y]
    M[i, s["doy"].to_numpy() - 1] = s["reason_code"].to_numpy()

colors = ["#2ca02c", "#d9d9d9", "#7f7f7f", "#d62728", "#ff7f0e", "#9467bd", "#8c564b", "#17becf", "#e377c2"]
fig, ax = plt.subplots(figsize=(14, max(4, 0.32 * len(years) + 1.5)))
ax.imshow(np.ma.masked_invalid(M), aspect="auto", cmap=ListedColormap(colors), vmin=-0.5, vmax=len(REASONS) - 0.5,
          interpolation="nearest")
ax.set_yticks(range(len(years))); ax.set_yticklabels(years, fontsize=8)
ax.set_xlabel("day of year"); ax.set_title("2. Verdict for every calendar day")
ax.legend(handles=[Patch(color=c, label=r) for c, r in zip(colors, REASONS)], bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout()

top = d.loc[~d["usable"], "reason"].value_counts()
save_fig("02", "usability_calendar", fig,
         f"{int(d['usable'].sum())} of {len(d)} days are usable. The most common reason for dropping a day is "
         f"'{top.index[0] if len(top) else 'none'}' ({int(top.iloc[0]) if len(top) else 0} days).")

## 2.4 Figure — the rule waterfall

**What this cell does.** Draws the before/after table from 2.2 as a bar chart: how many days survive each rule in turn.

**How to read it.** A big step down means that rule removes many days. If one rule dominates, loosening it (a setting in Part 0) would recover the most data — but only do that if the figure above shows those days look healthy.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(range(len(waterfall)), waterfall["days_left"], color="#3b7ddd")
for i, (v, drop) in enumerate(zip(waterfall["days_left"], waterfall["dropped_by_rule"])):
    ax.text(i, v, f"{v:,}\n(-{drop:,})" if i else f"{v:,}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(range(len(waterfall)))
ax.set_xticklabels([s.replace("after rule: ", "").replace("_", "\n") for s in waterfall["step"]], fontsize=8)
ax.set_ylabel("days left"); ax.set_title("3. Days remaining after each rule"); plt.tight_layout()

biggest = waterfall.iloc[1:].sort_values("dropped_by_rule", ascending=False).iloc[0]
save_fig("02", "rule_waterfall", fig,
         f"The rule that removes the most days is '{biggest['step'].replace('after rule: ', '')}' "
         f"({int(biggest['dropped_by_rule'])} days). Only {int(waterfall['days_left'].iloc[-1])} days survive all rules.")

## 2.5 Year scorecard and verdict

**What this cell does.** Summarises the day verdicts per year and gives each year a verdict:

* **USE** — at least 150 usable days spread over at least 9 different months.
* **USE_WITH_CAUTION** — at least 60 usable days.
* **DO_NOT_USE** — fewer than that.

It also records the longest unbroken run of usable days and how the data was lost (counts per reason).

**Why it matters.** This is the answer to "which part of the data can we use?" in one table. A month counts as covered if it has at least 10 usable days.

In [ ]:
def longest_true_run(s):
    return longest_run(s.to_numpy())

rows = []
for y, g in d.groupby("year"):
    u = g[g["usable"]]
    months_cov = int((u.groupby("month").size() >= 10).sum())
    n_use = len(u)
    verdict = ("USE" if n_use >= USE_MIN_DAYS and months_cov >= USE_MIN_MONTHS
               else "USE_WITH_CAUTION" if n_use >= CAUTION_MIN_DAYS else "DO_NOT_USE")
    reasons = g.loc[~g["usable"], "reason"].value_counts().to_dict()
    rows.append({"year": y, "calendar_days": len(g), "days_with_files": int(g["has_file"].sum()),
                 "usable_days": n_use, "usable_share_%": round(100 * n_use / len(g), 1),
                 "months_covered": months_cov, "longest_usable_run_days": longest_true_run(g["usable"]),
                 "verdict": verdict,
                 "conflict_day_share_%": round(100 * (g["conflict_minutes_any"] > 0).mean(), 1),
                 "poa_valid_days": int(g["poa_valid_day"].fillna(False).sum()),
                 "perf_index_tc_median": round(float(u["perf_index_tc"].median()), 4) if n_use else np.nan,
                 **{f"drop_{r}": int(reasons.get(r, 0)) for r in REASONS[1:]}})
scorecard = pd.DataFrame(rows)
display(scorecard)
print(scorecard["verdict"].value_counts().to_dict())

## 2.6 Figure — year scorecard

**What this cell does.** Plots usable days per year as a bar, coloured by the verdict (green = USE, orange = caution, red = do not use). Dashed lines show the two thresholds.

**How to read it.** The green bars are the years we can build the RUL history from. Check that 2023 sits where we expect given the snowstorm.

In [ ]:
vc = {"USE": "#2ca02c", "USE_WITH_CAUTION": "#ff9f1c", "DO_NOT_USE": "#d62728"}
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.bar(scorecard["year"], scorecard["usable_days"], color=scorecard["verdict"].map(vc))
ax.axhline(USE_MIN_DAYS, ls="--", c="grey", lw=1); ax.axhline(CAUTION_MIN_DAYS, ls=":", c="grey", lw=1)
ax.set_xlabel("year"); ax.set_ylabel("usable days"); ax.set_title("4. Usable days per year")
ax.legend(handles=[Patch(color=c, label=k) for k, c in vc.items()]); ax.grid(alpha=.3, axis="y"); plt.tight_layout()

vcount = scorecard["verdict"].value_counts()
yrs_use = scorecard.loc[scorecard["verdict"] == "USE", "year"].tolist()
save_fig("02", "year_scorecard", fig,
         f"{int(vcount.get('USE', 0))} years are USE, {int(vcount.get('USE_WITH_CAUTION', 0))} USE_WITH_CAUTION and "
         f"{int(vcount.get('DO_NOT_USE', 0))} DO_NOT_USE. Years to build on: {yrs_use if yrs_use else 'none yet'}.")

## 2.7 Figure — repeated readings (duplicates and conflicts) per year

**What this cell does.** For each year, shows the share of days that contain **conflicting** duplicates (same minute, same sensor, different values) and the share that contain only harmless exact copies.

**How to read it.** Our earlier tests found conflicts only in a window of 2020. This figure shows whether other years are affected too. Years with a lot of conflict need more caution because the conflict minutes are set to missing.

In [ ]:
yy = d[d["has_file"]].groupby("year").agg(
    conflict_days=("conflict_minutes_any", lambda s: (s > 0).mean() * 100),
    exact_dup_only_days=("exact_dup_groups", lambda s: (s > 0).mean() * 100))
fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(yy.index - 0.2, yy["exact_dup_only_days"], width=0.4, color="#9ecae1", label="days with exact duplicate copies (harmless)")
ax.bar(yy.index + 0.2, yy["conflict_days"], width=0.4, color="#d62728", label="days with conflicting copies")
ax.set_ylabel("% of days"); ax.set_xlabel("year"); ax.set_title("5. Repeated readings per year")
ax.legend(); ax.grid(alpha=.3, axis="y"); plt.tight_layout()

worst = yy["conflict_days"].idxmax()
save_fig("02", "conflicts_per_year", fig,
         f"Conflicting copies appear on {yy['conflict_days'].max():.1f}% of days in the worst year ({worst}) and on "
         f"{int((yy['conflict_days'] > 0).sum())} of {len(yy)} years in total. These readings are set to missing, never averaged.")

## 2.8 Figure — can we trust the irradiance sensor (POA), year by year?

**What this cell does.** For every year, shows (a) how often the POA sensor was logged in daylight and (b) the share of days where it passed all four validity checks: logged most of the day, daytime median between 20 and 1,400 W/m², night-time reading near zero, and it rises and falls together with power.

**Why it matters.** This answers *"what is behind the 50 % missing POA and can we use it?"* year by year. If the green bars are low, the sensor is not a usable reference in that year and we rely on the clear-sky model instead. If some years are high, we can use measured POA in those years (for example to double-check the model).

In [ ]:
hasf = d[d["has_file"]]
py = hasf.groupby("year").agg(cov=("poa_cov_day", "median"),
                              valid=("poa_valid_day", lambda s: s.fillna(False).mean() * 100),
                              night_med=("poa_night_median", "median"))
fig, ax = plt.subplots(figsize=(13, 4.2))
ax.bar(py.index - 0.2, py["cov"] * 100, width=0.4, color="#9ecae1", label="median daylight coverage %")
ax.bar(py.index + 0.2, py["valid"], width=0.4, color="#2ca02c", label="% of days passing all validity checks")
ax.set_ylim(0, 105); ax.set_ylabel("%"); ax.set_xlabel("year"); ax.set_title("6. Irradiance sensor (POA) quality by year")
ax.legend(loc="upper left"); ax.grid(alpha=.3, axis="y"); plt.tight_layout()

good_years = py.index[py["valid"] >= 80].tolist()
save_fig("02", "poa_validity_by_year", fig,
         f"The POA sensor passes the validity checks on at least 80% of days in {len(good_years)} of {len(py)} years "
         f"({good_years if good_years else 'none'}). Elsewhere it should not be used as the irradiance reference.")

## 2.9 Figure — zoom on the snowstorm period

**What this cell does.** For the first manually excluded period (February 2023), plots two things for each day in a window around it: the output relative to normal (`energy_vs_expected`) and the daytime air temperature. Days we exclude are shaded.

**How to read it.** If the damage really comes from snow, you should see output collapse on cold days and recover as the temperature rises. This also shows whether the automatic snow detector would have caught the storm without our manual exclusion.

In [ ]:
s0, e0, why0 = EXCLUDE_PERIODS[0] if EXCLUDE_PERIODS else (None, None, None)
if s0 is not None:
    w = d[d["date"].between(pd.Timestamp(s0) - pd.Timedelta("20D"), pd.Timestamp(e0) + pd.Timedelta("20D"))]
else:
    w = d.iloc[:0]
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
if len(w):
    axes[0].plot(w["date"], w["energy_vs_expected"], marker="o", ms=3)
    axes[0].axhline(SNOW_ENERGY_RATIO, ls="--", c="r", lw=1)
    axes[0].set_ylabel("output vs normal")
    axes[1].plot(w["date"], w["ambient_day_mean"], marker="o", ms=3, c="tab:orange")
    axes[1].axhline(SNOW_AMBIENT_C, ls="--", c="r", lw=1); axes[1].set_ylabel("daytime air temp (C)")
    for ax in axes:
        ax.axvspan(pd.Timestamp(s0), pd.Timestamp(e0), color="grey", alpha=.2)
        for x in w.loc[w["snow_episode"], "date"]:
            ax.axvline(x, color="tab:cyan", alpha=.25)
    axes[0].set_title("7. Snowstorm window (grey = manual exclusion, blue lines = auto-detected snow episode)")
plt.tight_layout()

if len(w):
    auto = int(w.loc[w["date"].between(pd.Timestamp(s0), pd.Timestamp(e0)), "snow_episode"].sum())
    total = int(w["date"].between(pd.Timestamp(s0), pd.Timestamp(e0)).sum())
    expl = (f"Inside the excluded window, the automatic snow detector also flags {auto} of {total} days, so the manual "
            f"exclusion is {'supported' if auto >= 0.3 * total else 'not strongly supported'} by the data on its own.")
else:
    expl = "No excluded period configured, nothing to show."
save_fig("02", "snowstorm_zoom", fig, expl)

## 2.10 Is there a degradation signal? (health indicator over time)

**What this cell does.** Takes the usable days, computes the monthly median of the temperature-corrected performance index, removes the normal seasonal pattern (by dividing each month by the long-run average of that calendar month), and fits a robust straight line through time (the Theil–Sen method, which ignores outliers). The slope is the estimated **loss per year**, and the line is extended to guess when the health would reach 80 % of its start.

**Why it matters.** RUL only makes sense if the panels visibly age. This is an *exploration*, not the final target: it tells us whether an "end of life" is even reachable inside the data. If the projected crossing is decades away, we will have very few examples of failure and must pick the target carefully (see Part 3).

In [ ]:
from scipy.stats import theilslopes

u = d[d["usable"] & d["perf_index_tc"].notna()]
monthly = (u.groupby(["year", "month"])["perf_index_tc"].agg(["median", "size"]).reset_index())
monthly = monthly[monthly["size"] >= 5]
monthly["rel"] = monthly["median"] / monthly.groupby("month")["median"].transform("mean")
monthly["t"] = monthly["year"] + (monthly["month"] - 0.5) / 12

deg = {"ok": False}
if len(monthly) >= 24 and monthly["year"].nunique() >= 3:
    t0 = monthly["t"].min()
    slope, icpt, lo, hi = theilslopes(monthly["rel"], monthly["t"] - t0)
    deg = {"ok": True, "t0": t0, "slope_per_year": slope, "icpt": icpt,
           "pct_per_year": 100 * slope / icpt, "pct_lo": 100 * lo / icpt, "pct_hi": 100 * hi / icpt,
           "years_to_eol": (EOL_FRACTION - 1) * icpt / slope if slope < 0 else np.inf}
    eol_txt = ("never (no decline found)" if not np.isfinite(deg["years_to_eol"])
               else "%.0f years" % deg["years_to_eol"])
    print("Estimated change: %.2f %% per year (range %.2f to %.2f)" % (deg["pct_per_year"], deg["pct_lo"], deg["pct_hi"]))
    print("Years of data in the fit:", monthly["year"].nunique(), "| monthly points:", len(monthly))
    print("Straight-line projection to %.0f%% of the starting level: %s" % (EOL_FRACTION * 100, eol_txt))
else:
    print("Not enough usable months/years to estimate a trend yet (need >= 24 months over >= 3 years).")

fig, axes = plt.subplots(2, 1, figsize=(13, 7))
axes[0].scatter(u["date"], u["perf_index_tc"], s=3, alpha=.3, label="usable days")
mm = monthly.assign(date=pd.to_datetime(dict(year=monthly["year"], month=monthly["month"], day=15)))
axes[0].plot(mm["date"], mm["median"], c="k", lw=1, label="monthly median")
axes[0].set_ylabel("performance index (W per W/m2)"); axes[0].legend(); axes[0].set_title("8a. Health indicator on usable days")
if deg["ok"]:
    axes[1].scatter(monthly["t"], monthly["rel"], s=14)
    tt = np.array([monthly["t"].min(), monthly["t"].max()])
    axes[1].plot(tt, deg["icpt"] + deg["slope_per_year"] * (tt - deg["t0"]), c="r", lw=1.5,
                 label="trend %.2f%%/yr" % deg["pct_per_year"])
    axes[1].legend()
axes[1].set_ylabel("seasonally adjusted (1 = average)"); axes[1].set_title("8b. Seasonal pattern removed, robust trend line")
plt.tight_layout()

if deg["ok"]:
    expl = ("After removing seasons, performance changes by about %.2f%% per year. A straight-line projection to %.0f%% "
            "of start gives %s; this matters for how the RUL target can be defined."
            % (deg["pct_per_year"], EOL_FRACTION * 100, "no crossing" if not np.isfinite(deg["years_to_eol"]) else "about %.0f years" % deg["years_to_eol"]))
else:
    expl = "Not enough usable months to estimate a degradation trend; widen usable data first."
save_fig("02", "health_indicator_trend", fig, expl)

## 2.11 Write the profile file for the review gate

**What this cell does.** Saves `data_profile.json`: a short, machine-readable summary of the daily table (size, missing values, duplicates, data type traits) plus a list of **leakage suspects** — columns that would secretly contain the answer if the RUL target is built from them.

**Why it matters.** The review gate reads this file. The leakage list is a warning for later notebooks: if the target is "time until the performance index drops to 80 %", then the performance-index columns must **not** also be used as inputs for the same moment in time.

In [ ]:
leak = ["perf_index_raw", "perf_index_tc", "yield_idx", "energy_vs_expected"]
profile = {
    "n_rows": int(len(d)), "n_cols": int(d.shape[1]),
    "unit_of_row": "one calendar day of System 50",
    "traits": {"has_datetime": True, "time_col": "date", "has_groups": False, "group_col": None,
               "small_data": bool(len(d) < 5000), "n_years": int(d["year"].nunique()),
               "imbalanced": False, "target_defined": False},
    "missing_pct": {c: round(float(d[c].isna().mean() * 100), 2) for c in d.columns},
    "duplicate_dates": int(d["date"].duplicated().sum()),
    "usable_days": int(d["usable"].sum()),
    "leakage_suspects": leak,
    "leakage_note": "These columns define the health indicator. If the RUL label is derived from them, they cannot also be inputs at the same time point.",
    "note": "The ml-pipeline guard library was not available, so this profile was produced by this notebook.",
}
(OUT_DIR / "data_profile.json").write_text(json.dumps(profile, indent=2, default=str))

keep_all = d.drop(columns=["reason_code"])
keep_all.to_parquet(DATA_DIR / "daily_quality.parquet", index=False)
scorecard.to_csv(DATA_DIR / "year_scorecard.csv", index=False)
print("Saved data_profile.json, daily_quality.parquet and year_scorecard.csv in", OUT_DIR.resolve())

# Part 3 — Define the prediction problem (step 3)

## 3.1 Draft contract for you to approve

**What this cell does.** Prints and saves a short draft of what "predict the RUL" would mean for System 50. Numbers from the earlier cells (usable years, degradation rate) are filled in automatically.

**Why it matters.** Everything after this point — cleaning, features, the split, the model — depends on this definition. It is a draft, not a decision. **You approve it (or change it) at Gate A.**

In [ ]:
usable_years = scorecard.loc[scorecard["verdict"].isin(["USE", "USE_WITH_CAUTION"]), "year"].tolist()
contract = {
    "status": "DRAFT - awaiting Gate A approval",
    "target": f"RUL_days = days from the prediction date until the smoothed, temperature-corrected, "
              f"clear-sky-normalised performance index first falls below {EOL_FRACTION:.0%} of its starting level.",
    "target_unit": "days",
    "prediction_unit": "one row per usable calendar day of System 50 (rul_ready_daily.parquet)",
    "population": f"usable days only; candidate years: {usable_years}",
    "prediction_time": "end of each usable day, using that day and earlier days only",
    "information_available": "daily features computed from the same day or earlier (no future data)",
    "objective": "regression of remaining days until end of life",
    "metric": "MAE in days (primary), plus error as a percentage of true RUL",
    "split_plan": "chronological by year (time-based train / validation / test); no random split",
    "constraints": ["only ONE physical system, so there is a single degradation path (generalisation to other panels cannot be tested)",
                    "irradiance sensor not trusted; health indicator relies on the clear-sky model",
                    "excluded periods: " + "; ".join(f"{a}..{b} ({w})" for a, b, w in EXCLUDE_PERIODS)],
    "open_risks": [],
}
if deg["ok"]:
    contract["degradation_estimate"] = f"{deg['pct_per_year']:.2f} %/year (range {deg['pct_lo']:.2f} to {deg['pct_hi']:.2f})"
    if (not np.isfinite(deg["years_to_eol"])) or deg["years_to_eol"] > 10:
        contract["open_risks"].append(
            f"End of life at {EOL_FRACTION:.0%} may never be reached inside the data "
            "(projection: " + ("no decline" if not np.isfinite(deg["years_to_eol"]) else "%.0f years" % deg["years_to_eol"]) + "). "
            "Labels would be censored. Options: a milder threshold (e.g. 90-95 %), a degradation-rate target, or a different EOL definition.")
contract["open_risks"].append("A single system gives one run-to-threshold trajectory, so a model can only be validated on later years of the same panel.")

(OUT_DIR / "problem_contract_draft.json").write_text(json.dumps(contract, indent=2, default=str))
print(json.dumps(contract, indent=2, default=str))

## 3.2 Review-gate summary and the Gate A switch

**What this cell does.** Writes (or updates) `PIPELINE.md`, the progress checklist, and prints the Gate A summary: what was done, what was found, what comes next. Any gate approval lines already in the file are kept.

**How Gate A works.** Read the figures and the contract above. If you agree, change `GATE_A_APPROVED = False` to `True` in the Part 0 settings cell and tell the assistant so the approval is recorded as a dated line in `PIPELINE.md`. Part 4 refuses to write the RUL-ready files while the switch is `False`.

In [ ]:
STEPS = ["1. Data inspection", "2. Exploratory data analysis", "3. Define the prediction problem",
         "4. Data cleaning", "5. Data engineering", "6. Train / validation / test split",
         "7. Feature engineering", "8. Preprocessing", "9. Baseline model", "10. Model training",
         "11. Hyperparameter tuning", "12. Model evaluation", "13. Error analysis", "14. Final test",
         "15. Deployment", "16. Monitoring + retraining plan"]

def write_pipeline_md(status, results):
    p = OUT_DIR / "PIPELINE.md"
    kept = []
    if p.exists():
        kept = [l for l in p.read_text().splitlines() if l.startswith("- Gate ") or l.startswith("- Override")]
    lines = ["# PIPELINE - System 50 RUL", "", "## Steps", ""]
    for s in STEPS:
        lines.append(f"- [{'x' if status.get(s) == 'done' else ' '}] {s} - {status.get(s, 'todo')}"
                     + (f" - {results[s]}" if s in results else ""))
    lines += ["", "## Decisions and rules", "",
              "- Exact duplicate copies collapsed to one; conflicting copies set to missing (never averaged).",
              f"- Day usable if: daylight power coverage >= {MIN_DAYLIGHT_COV:.0%}, longest gap <= {MAX_DAYLIGHT_GAP_MIN} min, "
              f"conflict share <= {MAX_CONFLICT_SHARE:.0%}, >= {MIN_CLEAR_MINUTES} clear-sky minutes, not in snow episode, not manually excluded.",
              "- Manual exclusions: " + "; ".join(f"{a}..{b} ({w})" for a, b, w in EXCLUDE_PERIODS),
              "", "## Figures", ""]
    lines += [f"- Figure {f['file']}: {f['explanation']}" for f in FIG_LOG]
    lines += ["", "## Gates", ""] + (kept if kept else ["- Gate A: todo"])
    p.write_text("\n".join(lines) + "\n")

write_pipeline_md(
    {"1. Data inspection": "done", "2. Exploratory data analysis": "done", "3. Define the prediction problem": "in progress"},
    {"1. Data inspection": f"{len(d):,} calendar days profiled",
     "2. Exploratory data analysis": f"{int(d['usable'].sum()):,} usable days; verdicts {scorecard['verdict'].value_counts().to_dict()}",
     "3. Define the prediction problem": "draft contract written, waiting for Gate A"})

print("=== GATE A SUMMARY ===")
print("Done      : inspected every file; profiled every day; scored every year; drafted the RUL contract.")
print(f"Found     : {int(d['usable'].sum()):,} usable days of {len(d):,}. Year verdicts: {scorecard['verdict'].value_counts().to_dict()}")
print("Decided   : conflicting duplicates -> missing (not averaged); Feb 2023 excluded by hand; POA sensor not relied on.")
print("Next      : cleaning + data engineering (writes rul_ready_daily.parquet), then split, features, preprocessing.")
print("Question  : do you approve the contract in problem_contract_draft.json? If yes, set GATE_A_APPROVED = True.")
print("Gate A is currently:", "APPROVED" if GATE_A_APPROVED else "NOT approved")

# Part 4 — Cleaning and data engineering (steps 4 and 5) — locked until Gate A

Everything below only runs when `GATE_A_APPROVED = True`. Until then each cell prints a message and does nothing.

## 4.1 Build the RUL-ready daily table

**What this cell does.** Keeps only the **usable** days and only the columns the next notebooks need, with clear names and units. It adds a `days_since_start` column (time since the first usable day), checks the table is clean (no duplicate dates, sorted, valid values), and saves it as `rul_ready_daily.parquet`. A second file lists the column meanings.

**Why it matters.** This is the hand-over file. It has **one row per usable day** and no target yet — the RUL label is created in the next notebook once Gate A is approved. We deliberately keep `date` and `year` so the next notebook can split by time.

In [ ]:
RUL_COLS = {
    "date": "calendar date (local clock, UTC-7, no daylight-saving shift)",
    "year": "calendar year",
    "days_since_start": "days since the first usable day",
    "perf_index_tc": "health indicator: DC power / clear-sky plane-of-array irradiance, corrected to 25 C module temperature",
    "perf_index_raw": "same, without the temperature correction",
    "energy_vs_expected": "daily yield relative to the surrounding-year 90th percentile (1 = normal day)",
    "dc_energy_kwh_obs": "DC energy of the day, kWh (observed minutes only)",
    "cs_energy_kwh_m2": "modelled clear-sky plane-of-array energy of the day, kWh/m2",
    "dc_peak_w": "99.9th percentile of power in the day, W",
    "module_temp_day_mean": "mean module temperature in daylight, C",
    "ambient_day_mean": "mean air temperature in daylight, C",
    "ambient_min": "lowest air temperature of the day, C",
    "clear_minutes": "number of clear-sky minutes used for the performance index",
    "cov_day_power": "share of daylight minutes with a power reading (0-1)",
    "conflict_share_day": "share of daylight minutes touched by conflicting duplicates (0-1)",
    "poa_valid_day": "True if the irradiance sensor passed all validity checks that day",
}

def build_rul_ready():
    out = d[d["usable"]].copy()
    out["days_since_start"] = (out["date"] - out["date"].min()).dt.days
    out = out[list(RUL_COLS)].sort_values("date").reset_index(drop=True)
    assert out["date"].is_unique, "duplicate dates in output"
    assert out["date"].is_monotonic_increasing
    assert out["perf_index_tc"].notna().mean() > 0.5, "too few valid performance-index values"
    assert (out["cov_day_power"] >= MIN_DAYLIGHT_COV).all()
    return out

if GATE_A_APPROVED:
    rul_ready = build_rul_ready()
    rul_ready.to_parquet(DATA_DIR / "rul_ready_daily.parquet", index=False)
    pd.Series(RUL_COLS, name="meaning").to_frame().to_csv(DATA_DIR / "rul_ready_daily_columns.csv")
    print("Saved rul_ready_daily.parquet:", rul_ready.shape)
    display(rul_ready.groupby("year").size().to_frame("usable_days").T)
    display(rul_ready.head())
else:
    print("Gate A not approved - rul_ready_daily.parquet NOT written. Review Part 3, then set GATE_A_APPROVED = True.")

## 4.2 Optional: minute-level cleaned data for chosen years

**What this cell does.** For the years you list in `EXPORT_MINUTE_YEARS`, re-reads each **usable** day and writes the cleaned one-minute table: one row per minute, one column per sensor, conflicting readings left as missing, plus three helper columns: `conflict_minute`, `sun_elevation` and `poa_clearsky`. Files are written as one Parquet file per day under `data/minute_level/year=YYYY/`.

**Why it matters.** Some later steps (e.g. detailed feature engineering) may need minute-level data. We do not export every year by default because the full history is very large. Leave the list empty to skip.

In [ ]:
EXPORT_MINUTE_YEARS = []   # e.g. [2018, 2019]. Empty = skip.

def export_minute_level(years):
    sel = d[d["usable"] & d["year"].isin(years)][["date"]].merge(days[["date", "paths"]], on="date")
    def one(t):
        date, paths = t
        wide, info, idx_utc, _ = clean_day(read_day(paths), date)
        out = wide.copy()
        out["conflict_minute"] = info["conflict_minute"].to_numpy()
        out["sun_elevation"] = info["elev"].to_numpy()
        out["poa_clearsky"] = clearsky_poa(info, idx_utc).to_numpy()
        out.index.name = "measured_on"
        part = DATA_DIR / "minute_level" / f"year={date.year}"
        part.mkdir(parents=True, exist_ok=True)
        out.reset_index().to_parquet(part / f"{date:%Y-%m-%d}.parquet", index=False)
        return len(out)
    with ThreadPoolExecutor(WORKERS) as ex:
        n = list(ex.map(one, zip(sel["date"], sel["paths"])))
    return len(n), sum(n)

if GATE_A_APPROVED and EXPORT_MINUTE_YEARS:
    nd, nr = export_minute_level(EXPORT_MINUTE_YEARS)
    print(f"Wrote {nd} day files, {nr:,} minute rows")
elif not GATE_A_APPROVED:
    print("Gate A not approved - minute-level export skipped.")
else:
    print("EXPORT_MINUTE_YEARS is empty - minute-level export skipped.")

## 4.3 Manifest (settings and results in one file)

**What this cell does.** Saves `manifest.json` with every setting used, the library versions, how many days passed each rule, the year verdicts, and the list of files written. It also updates `PIPELINE.md` to mark steps 4 and 5 as done.

**Why it matters.** Months from now you can see exactly how `rul_ready_daily.parquet` was made, and re-create it. If any setting changes, the cache hash in the file names changes and the profile is recomputed.

In [ ]:
if GATE_A_APPROVED:
    manifest = {
        "created": pd.Timestamp.now().isoformat(),
        "system_id": SYSTEM_ID,
        "settings": {k: v for k, v in globals().items()
                     if k.isupper() and isinstance(v, (int, float, str, list, tuple, bool, type(None)))
                     and k not in ("BUCKET", "BASE", "KEYS", "REASONS", "STEPS", "RUL_COLS", "CH_KEY", "FIG_LOG", "CFG_KEYS")},
        "library_versions": {"pandas": pd.__version__, "numpy": np.__version__, "pvlib": pvlib.__version__},
        "waterfall": waterfall.to_dict("records"),
        "year_verdicts": scorecard[["year", "usable_days", "verdict"]].to_dict("records"),
        "files": sorted(str(p.relative_to(OUT_DIR)) for p in OUT_DIR.rglob("*") if p.is_file() and "cache" not in p.parts),
    }
    (DATA_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
    write_pipeline_md(
        {"1. Data inspection": "done", "2. Exploratory data analysis": "done", "3. Define the prediction problem": "done",
         "4. Data cleaning": "done", "5. Data engineering": "done"},
        {"4. Data cleaning": "duplicates collapsed, conflicts -> missing, bad/snow/manual days labelled and removed",
         "5. Data engineering": f"rul_ready_daily.parquet: {len(rul_ready):,} rows"})
    print("manifest.json written; PIPELINE.md updated")
else:
    print("Gate A not approved - manifest not written.")

## How the next notebooks use these files

**What this cell does.** Shows the two lines a later notebook needs to load the results.

**Reminder for those notebooks:** split by **time** (e.g. earlier years for training, later years for validation and test), never randomly; build the RUL label from `perf_index_tc` using the approved contract; and do not use `perf_index_*` or `energy_vs_expected` from the same day as inputs if the label is derived from them (leakage).

In [ ]:
# In the next notebook:
# rul = pd.read_parquet(".../ml_pipeline/data/rul_ready_daily.parquet")      # usable days only
# quality = pd.read_parquet(".../ml_pipeline/data/daily_quality.parquet")    # all days + why each was kept/dropped
# years = pd.read_csv(".../ml_pipeline/data/year_scorecard.csv")             # year verdicts
print("Pipeline notebook finished. Gate A is", "APPROVED" if GATE_A_APPROVED else "NOT yet approved")